<a href="https://colab.research.google.com/github/keswong/phd_listing_repo/blob/main/3_10_2_Comparing_Diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pingouin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 10.5 MB/s eta 0:00:00


# **Social-Cognitive Dimension**

## *Predict and Teacher*

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    chi2,
    norm
)

data = {
    'Cognitive_Coded':   [20, 49, 18, 0, 0, 2, 0, 23, 8, 32],
    'Cognitive_Predict': [0, 56, 21, 0, 0, 0, 0, 22, 8, 45],
    'Cognitive_Teacher': [4, 48, 44, 3, 6, 33, 9, 0, 0, 5]
}

index = [
    'SS1', 'SS2', 'SS3', 'SS4', 'SS5',
    'SS6', 'SS10', 'SC1', 'SC2', 'OTHER'
]

df = pd.DataFrame(data, index=index)

print("\n==============================")
print("FULL CONTINGENCY TABLE")
print("==============================")
print(df)

chi2_stat, p, dof, expected = chi2_contingency(df)

print("\n==============================")
print("OVERALL CHI-SQUARE TEST")
print("==============================")

print(f"Chi-square = {chi2_stat:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"P-value = {p:.6e}")

pair_df = df[['Cognitive_Predict', 'Cognitive_Teacher']]
chi2_pair, p_pair, dof_pair, expected_pair = chi2_contingency(pair_df)

print("\n==============================")
print("POST-HOC: Predict vs Teacher")
print("==============================")

print(pair_df)

print(f"\nChi-square = {chi2_pair:.4f}")
print(f"Degrees of freedom = {dof_pair}")
print(f"P-value = {p_pair:.6e}")

# =========================================================
# STANDARDIZED RESIDUALS
# =========================================================
print(expected_pair)

print(pair_df)
residuals = (
    pair_df - expected_pair
) / np.sqrt(expected_pair)

residuals_df = pd.DataFrame(
    residuals,
    index=pair_df.index,
    columns=pair_df.columns
)

print("\n==============================")
print("STANDARDIZED RESIDUALS")
print("==============================")

print(residuals_df.round(2))

print("\n==============================")
print("CELLS WITH LARGE RESIDUALS")
print("==============================")

for row in residuals_df.index:
    for col in residuals_df.columns:
        r = residuals_df.loc[row, col]

        if abs(r) >= 2:
            direction = "higher than expected" if r > 0 else "lower than expected"

            print(
                f"{row} in {col}: residual = {r:.2f} "
                f"({direction})"
            )

print("\n==============================")
print("POST-HOC PROPORTION TESTS")
print("==============================")

predict_total = pair_df['Cognitive_Predict'].sum()
teacher_total = pair_df['Cognitive_Teacher'].sum()

results = []

for skill in pair_df.index:

    count1 = pair_df.loc[skill, 'Cognitive_Predict']
    count2 = pair_df.loc[skill, 'Cognitive_Teacher']
    count3 = residuals.loc[skill, 'Cognitive_Predict']
    count4 = residuals.loc[skill, 'Cognitive_Teacher']

    p1 = count1 / predict_total
    p2 = count2 / teacher_total

    pooled = (count1 + count2) / (predict_total + teacher_total)

    se = np.sqrt(
        pooled * (1 - pooled) *
        ((1 / predict_total) + (1 / teacher_total))
    )

    if se == 0:
        z = 0
        pval = 1
    else:
        z = (p1 - p2) / se
        pval = 2 * (1 - norm.cdf(abs(z)))

    results.append([
        skill,
        count1,
        count2,
        count3,
        count4,
        z,
        pval
    ])

posthoc_df = pd.DataFrame(
    results,
    columns=[
        'Subskill',
        'Predict_Count',
        'Teacher_Count',
        'Standardised Residual Predict_Count',
        'Standardised Residual Teacher_Count',
        'Z',
        'p'
    ]
)


# Bonferroni correction
posthoc_df['Bonferroni_p'] = (
    posthoc_df['p'] * len(posthoc_df)
).clip(upper=1)

posthoc_df['Significant'] = (
    posthoc_df['Bonferroni_p'] < 0.05
)


posthoc_df['Standardised Residual Predict_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Predict_Count'],
    errors='coerce'
)

posthoc_df['Standardised Residual Predict_Count'] = posthoc_df['Standardised Residual Predict_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

posthoc_df['Standardised Residual Teacher_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Teacher_Count'],
    errors='coerce'
)

posthoc_df['Standardised Residual Teacher_Count'] = posthoc_df['Standardised Residual Teacher_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

posthoc_df['p'] = posthoc_df['p'].apply(lambda x: f"{x:.3g}")
posthoc_df['Z'] = posthoc_df['Z'].apply(lambda x: f"{x:.2f}")
posthoc_df['Bonferroni_p'] = posthoc_df['Bonferroni_p'].apply(lambda x: f"{x:.3g}")

posthoc_df = posthoc_df.drop(columns=['Predict_Count', 'Teacher_Count'], errors='ignore')
print(posthoc_df)

print("\n==============================")
print("APA STYLE REPORT")
print("==============================")

if p_pair < 0.001:
    p_text = "p < .001"
else:
    p_text = f"p = {p_pair:.6e}"

print(
    f"A post-hoc Pearson chi-square test comparing "
    f"Cognitive_Predict and Cognitive_Teacher "
    f"showed that subskill distributions differed "
    f"significantly, χ²({dof_pair}) = "
    f"{chi2_pair:.2f}, {p }{p_text}."
)


FULL CONTINGENCY TABLE
       Cognitive_Coded  Cognitive_Predict  Cognitive_Teacher
SS1                 20                  0                  4
SS2                 49                 56                 48
SS3                 18                 21                 44
SS4                  0                  0                  3
SS5                  0                  0                  6
SS6                  2                  0                 33
SS10                 0                  0                  9
SC1                 23                 22                  0
SC2                  8                  8                  0
OTHER               32                 45                  5

OVERALL CHI-SQUARE TEST
Chi-square = 199.0541
Degrees of freedom = 18
P-value = 1.548650e-32

POST-HOC: Predict vs Teacher
       Cognitive_Predict  Cognitive_Teacher
SS1                    0                  4
SS2                   56                 48
SS3                   21                 44
SS4  

## Coded and Teacher

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    chi2,
    norm
)

data = {
    'Cognitive_Coded':   [20, 49, 18, 0, 0, 2, 0, 23, 8, 32],
    'Cognitive_Predict': [0, 56, 21, 0, 0, 0, 0, 22, 8, 45],
    'Cognitive_Teacher': [4, 48, 44, 3, 6, 33, 9, 0, 0, 5]
}

index = [
    'SS1', 'SS2', 'SS3', 'SS4', 'SS5',
    'SS6', 'SS10', 'SC1', 'SC2', 'OTHER'
]

df = pd.DataFrame(data, index=index)

print("\n==============================")
print("FULL CONTINGENCY TABLE")
print("==============================")
print(df)

chi2_stat, p, dof, expected = chi2_contingency(df)

print("\n==============================")
print("OVERALL CHI-SQUARE TEST")
print("==============================")

print(f"Chi-square = {chi2_stat:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"P-value = {p:.6e}")


pair_df = df[['Cognitive_Coded', 'Cognitive_Teacher']]

chi2_pair, p_pair, dof_pair, expected_pair = chi2_contingency(pair_df)


print("\n==============================")
print("POST-HOC: Coded vs Teacher")
print("==============================")

print(pair_df)

print(f"\nChi-square = {chi2_pair:.4f}")
print(f"Degrees of freedom = {dof_pair}")
print(f"P-value = {p_pair:.6e}")

print(expected_pair)


residuals = (
    pair_df - expected_pair
) / np.sqrt(expected_pair)

residuals_df = pd.DataFrame(
    residuals,
    index=pair_df.index,
    columns=pair_df.columns
)

print("\n==============================")
print("STANDARDIZED RESIDUALS")
print("==============================")

print(residuals_df.round(2))

print("\n==============================")
print("CELLS WITH LARGE RESIDUALS")
print("==============================")

for row in residuals_df.index:
    for col in residuals_df.columns:
        r = residuals_df.loc[row, col]

        if abs(r) >= 2:
            direction = "higher than expected" if r > 0 else "lower than expected"

            print(
                f"{row} in {col}: residual = {r:.2f} "
                f"({direction})"
            )

print("\n==============================")
print("POST-HOC PROPORTION TESTS")
print("==============================")

predict_total = pair_df['Cognitive_Coded'].sum()
teacher_total = pair_df['Cognitive_Teacher'].sum()

results = []

for skill in pair_df.index:

    count1 = pair_df.loc[skill, 'Cognitive_Coded']
    count2 = pair_df.loc[skill, 'Cognitive_Teacher']
    count3 = residuals.loc[skill, 'Cognitive_Coded']
    count4 = residuals.loc[skill, 'Cognitive_Teacher']

    p1 = count1 / predict_total
    p2 = count2 / teacher_total

    pooled = (count1 + count2) / (predict_total + teacher_total)

    se = np.sqrt(
        pooled * (1 - pooled) *
        ((1 / predict_total) + (1 / teacher_total))
    )

    if se == 0:
        z = 0
        pval = 1
    else:
        z = (p1 - p2) / se
        pval = 2 * (1 - norm.cdf(abs(z)))

    results.append([
        skill,
        count1,
        count2,
        count3,
        count4,
        z,
        pval
    ])

posthoc_df = pd.DataFrame(
    results,
    columns=[
        'Subskill',
        'Coded_Count',
        'Teacher_Count',
        'Standardised Residual Coded_Count',
        'Standardised Residual Teacher_Count',
        'Z',
        'p'
    ]
)

# Bonferroni correction
posthoc_df['Bonferroni_p'] = (
    posthoc_df['p'] * len(posthoc_df)
).clip(upper=1)

posthoc_df['Significant'] = (
    posthoc_df['Bonferroni_p'] < 0.05
)

posthoc_df['Standardised Residual Coded_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Coded_Count'],
    errors='coerce'
)

posthoc_df['Standardised Residual Coded_Count'] = posthoc_df['Standardised Residual Coded_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

posthoc_df['Standardised Residual Teacher_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Teacher_Count'],
    errors='coerce'
)

posthoc_df['Standardised Residual Teacher_Count'] = posthoc_df['Standardised Residual Teacher_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)


posthoc_df['p'] = posthoc_df['p'].apply(lambda x: f"{x:.3g}")
posthoc_df['Z'] = posthoc_df['Z'].apply(lambda x: f"{x:.2f}")
posthoc_df['Bonferroni_p'] = posthoc_df['Bonferroni_p'].apply(lambda x: f"{x:.3g}")

posthoc_df = posthoc_df.drop(columns=['Predict_Count', 'Teacher_Count'], errors='ignore')
print(posthoc_df)

print("\n==============================")
print("APA STYLE REPORT")
print("==============================")

if p_pair < 0.001:
    p_text = "p < .001"
else:
    p_text = f"p = {p_pair:.6e}"

print(
    f"A post-hoc Pearson chi-square test comparing "
    f"Cognitive_Coded and Cognitive_Teacher "
    f"showed that subskill distributions differed "
    f"significantly, χ²({dof_pair}) = "
    f"{chi2_pair:.2f}, {p }{p_text}."
)


FULL CONTINGENCY TABLE
       Cognitive_Coded  Cognitive_Predict  Cognitive_Teacher
SS1                 20                  0                  4
SS2                 49                 56                 48
SS3                 18                 21                 44
SS4                  0                  0                  3
SS5                  0                  0                  6
SS6                  2                  0                 33
SS10                 0                  0                  9
SC1                 23                 22                  0
SC2                  8                  8                  0
OTHER               32                 45                  5

OVERALL CHI-SQUARE TEST
Chi-square = 199.0541
Degrees of freedom = 18
P-value = 1.548650e-32

POST-HOC: Coded vs Teacher
       Cognitive_Coded  Cognitive_Teacher
SS1                 20                  4
SS2                 49                 48
SS3                 18                 44
SS4            

# **Affective Dimension**

## *Predict and Teacher*

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    chi2,
    norm
)

data = {
    'Affective_Coded':   [0, 18, 1],
    'Affective_Predict': [0, 19, 0],
    'Affective_Teacher': [58, 42, 10]
}

index = ['AS1', 'AS2', 'AS3']

df = pd.DataFrame(data, index=index)

print("\n==============================")
print("FULL CONTINGENCY TABLE")
print("==============================")
print(df)

# =========================================================
# OVERALL CHI-SQUARE
# =========================================================

chi2_stat, p, dof, expected = chi2_contingency(df)

print("\n==============================")
print("OVERALL CHI-SQUARE TEST")
print("==============================")

print(f"Chi-square = {chi2_stat:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"P-value = {p:.6e}")

# =========================================================
# POST-HOC:
# PREDICT vs TEACHER
# =========================================================

pair_df = df[['Affective_Predict', 'Affective_Teacher']]

chi2_pair, p_pair, dof_pair, expected_pair = chi2_contingency(pair_df)


print("\n==============================")
print("POST-HOC: Prdict vs Teacher")
print("==============================")


print(pair_df)

print(f"\nChi-square = {chi2_pair:.4f}")
print(f"Degrees of freedom = {dof_pair}")
print(f"P-value = {p_pair:.6e}")

print(expected_pair)
# =========================================================
# STANDARDIZED RESIDUALS
# =========================================================

residuals = (
    pair_df - expected_pair
) / np.sqrt(expected_pair)

residuals_df = pd.DataFrame(
    residuals,
    index=pair_df.index,
    columns=pair_df.columns
)

print("\n==============================")
print("STANDARDIZED RESIDUALS")
print("==============================")

print(residuals_df.round(2))

# =========================================================
# INTERPRET LARGE RESIDUALS
# =========================================================

print("\n==============================")
print("CELLS WITH LARGE RESIDUALS")
print("==============================")

for row in residuals_df.index:
    for col in residuals_df.columns:
        r = residuals_df.loc[row, col]

        if abs(r) >= 2:
            direction = "higher than expected" if r > 0 else "lower than expected"

            print(
                f"{row} in {col}: residual = {r:.2f} "
                f"({direction})"
            )

# =========================================================
# POST-HOC PROPORTION TESTS
# =========================================================
# Compare proportions for each subskill separately
# Bonferroni corrected
# =========================================================

print("\n==============================")
print("POST-HOC PROPORTION TESTS")
print("==============================")

predict_total = pair_df['Affective_Predict'].sum()
teacher_total = pair_df['Affective_Teacher'].sum()

results = []

for skill in pair_df.index:

    count1 = pair_df.loc[skill, 'Affective_Predict']
    count2 = pair_df.loc[skill, 'Affective_Teacher']
    count3 = residuals.loc[skill, 'Affective_Predict']
    count4 = residuals.loc[skill, 'Affective_Teacher']

    p1 = count1 / predict_total
    p2 = count2 / teacher_total

    pooled = (count1 + count2) / (predict_total + teacher_total)

    se = np.sqrt(
        pooled * (1 - pooled) *
        ((1 / predict_total) + (1 / teacher_total))
    )

    # Avoid divide-by-zero
    if se == 0:
        z = 0
        pval = 1
    else:
        z = (p1 - p2) / se
        pval = 2 * (1 - norm.cdf(abs(z)))

    results.append([
        skill,
        count1,
        count2,
        count3,
        count4,
        z,
        pval
    ])

posthoc_df = pd.DataFrame(
    results,
    columns=[
        'Subskill',
        'Predict_Count',
        'Teacher_Count',
        'Standardised Residual Predict_Count',
        'Standardised Residual Teacher_Count',
        'Z',
        'p'
    ]
)

# Bonferroni correction
posthoc_df['Bonferroni_p'] = (
    posthoc_df['p'] * len(posthoc_df)
).clip(upper=1)

posthoc_df['Significant'] = (
    posthoc_df['Bonferroni_p'] < 0.05
)


# 1. Convert the column from text to numbers, turning errors into NaN
posthoc_df['Standardised Residual Predict_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Predict_Count'],
    errors='coerce'
)

# 2. Now you can safely apply the 3 significant figures formatting
posthoc_df['Standardised Residual Predict_Count'] = posthoc_df['Standardised Residual Predict_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

# 1. Convert the column from text to numbers, turning errors into NaN
posthoc_df['Standardised Residual Teacher_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Teacher_Count'],
    errors='coerce'
)

# 2. Now you can safely apply the 3 significant figures formatting
posthoc_df['Standardised Residual Teacher_Count'] = posthoc_df['Standardised Residual Teacher_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

# 2. Format 'p' to scientific notation (3 significant figures -> '.2e')
posthoc_df['p'] = posthoc_df['p'].apply(lambda x: f"{x:.3g}")

# 3. Format 'Z' to 3 significant figures
# (Note: Use '.3g' for general significant figures, or '.2f' if you want exactly 2 decimal places)
posthoc_df['Z'] = posthoc_df['Z'].apply(lambda x: f"{x:.2f}")

# 3. Format 'Z' to 3 significant figures
# (Note: Use '.3g' for general significant figures, or '.2f' if you want exactly 2 decimal places)
posthoc_df['Bonferroni_p'] = posthoc_df['Bonferroni_p'].apply(lambda x: f"{x:.3g}")

posthoc_df = posthoc_df.drop(columns=['Predict_Count', 'Teacher_Count'], errors='ignore')
print(posthoc_df)

# =========================================================
# APA STYLE
# =========================================================

print("\n==============================")
print("APA STYLE REPORT")
print("==============================")

if p_pair < 0.001:
    p_text = "p < .001"
else:
    p_text = f"p = {p_pair:.6e}"

print(
    f"A post-hoc Pearson chi-square test comparing "
    f"Affective_Predict and Affective_Teacher "
    f"showed that subskill distributions differed "
    f"significantly, χ²({dof_pair}) = "
    f"{chi2_pair:.2f}, {p }{p_text}."
)


FULL CONTINGENCY TABLE
     Affective_Coded  Affective_Predict  Affective_Teacher
AS1                0                  0                 58
AS2               18                 19                 42
AS3                1                  0                 10

OVERALL CHI-SQUARE TEST
Chi-square = 40.5603
Degrees of freedom = 4
P-value = 3.314418e-08

POST-HOC: Prdict vs Teacher
     Affective_Predict  Affective_Teacher
AS1                  0                 58
AS2                 19                 42
AS3                  0                 10

Chi-square = 24.8387
Degrees of freedom = 2
P-value = 4.039563e-06
[[ 8.54263566 49.45736434]
 [ 8.98449612 52.01550388]
 [ 1.47286822  8.52713178]]

STANDARDIZED RESIDUALS
     Affective_Predict  Affective_Teacher
AS1              -2.92               1.21
AS2               3.34              -1.39
AS3              -1.21               0.50

CELLS WITH LARGE RESIDUALS
AS1 in Affective_Predict: residual = -2.92 (lower than expected)
AS2 in Affective

## *Coded and Teacher*

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    chi2,
    norm
)

data = {
    'Affective_Coded':   [0, 18, 1],
    'Affective_Predict': [0, 19, 0],
    'Affective_Teacher': [58, 42, 10]
}

index = ['AS1', 'AS2', 'AS3']

df = pd.DataFrame(data, index=index)

print("\n==============================")
print("FULL CONTINGENCY TABLE")
print("==============================")
print(df)

chi2_stat, p, dof, expected = chi2_contingency(df)

print("\n==============================")
print("OVERALL CHI-SQUARE TEST")
print("==============================")

print(f"Chi-square = {chi2_stat:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"P-value = {p:.6e}")

pair_df = df[['Affective_Coded', 'Affective_Teacher']]

chi2_pair, p_pair, dof_pair, expected_pair = chi2_contingency(pair_df)

print("\n==============================")
print("POST-HOC: Coded vs Teacher")
print("==============================")

print(pair_df)

print(f"\nChi-square = {chi2_pair:.4f}")
print(f"Degrees of freedom = {dof_pair}")
print(f"P-value = {p_pair:.6e}")

print(expected_pair)

residuals = (
    pair_df - expected_pair
) / np.sqrt(expected_pair)

residuals_df = pd.DataFrame(
    residuals,
    index=pair_df.index,
    columns=pair_df.columns
)

print("\n==============================")
print("STANDARDIZED RESIDUALS")
print("==============================")

print(residuals_df.round(2))

print("\n==============================")
print("CELLS WITH LARGE RESIDUALS")
print("==============================")

for row in residuals_df.index:
    for col in residuals_df.columns:
        r = residuals_df.loc[row, col]

        if abs(r) >= 2:
            direction = "higher than expected" if r > 0 else "lower than expected"

            print(
                f"{row} in {col}: residual = {r:.2f} "
                f"({direction})"
            )

print("\n==============================")
print("POST-HOC PROPORTION TESTS")
print("==============================")

predict_total = pair_df['Affective_Coded'].sum()
teacher_total = pair_df['Affective_Teacher'].sum()

results = []

for skill in pair_df.index:

    count1 = pair_df.loc[skill, 'Affective_Coded']
    count2 = pair_df.loc[skill, 'Affective_Teacher']
    count3 = residuals.loc[skill, 'Affective_Coded']
    count4 = residuals.loc[skill, 'Affective_Teacher']

    p1 = count1 / predict_total
    p2 = count2 / teacher_total

    pooled = (count1 + count2) / (predict_total + teacher_total)

    se = np.sqrt(
        pooled * (1 - pooled) *
        ((1 / predict_total) + (1 / teacher_total))
    )

    if se == 0:
        z = 0
        pval = 1
    else:
        z = (p1 - p2) / se
        pval = 2 * (1 - norm.cdf(abs(z)))

    results.append([
        skill,
        count1,
        count2,
        count3,
        count4,
        z,
        pval
    ])

posthoc_df = pd.DataFrame(
    results,
    columns=[
        'Subskill',
        'Coded_Count',
        'Teacher_Count',
        'Standardised Residual Coded_Count',
        'Standardised Residual Teacher_Count',
        'Z',
        'p'
    ]
)

# Bonferroni correction
posthoc_df['Bonferroni_p'] = (
    posthoc_df['p'] * len(posthoc_df)
).clip(upper=1)

posthoc_df['Significant'] = (
    posthoc_df['Bonferroni_p'] < 0.05
)



posthoc_df['Standardised Residual Coded_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Coded_Count'],
    errors='coerce'
)


posthoc_df['Standardised Residual Coded_Count'] = posthoc_df['Standardised Residual Coded_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

posthoc_df['Standardised Residual Teacher_Count'] = pd.to_numeric(
    posthoc_df['Standardised Residual Teacher_Count'],
    errors='coerce'
)

posthoc_df['Standardised Residual Teacher_Count'] = posthoc_df['Standardised Residual Teacher_Count'].apply(
    lambda x: f"{x:.2f}" if pd.notnull(x) else x
)

posthoc_df['p'] = posthoc_df['p'].apply(lambda x: f"{x:.3g}")

posthoc_df['Z'] = posthoc_df['Z'].apply(lambda x: f"{x:.2f}")

posthoc_df['Bonferroni_p'] = posthoc_df['Bonferroni_p'].apply(lambda x: f"{x:.3g}")

posthoc_df = posthoc_df.drop(columns=['Predict_Count', 'Teacher_Count'], errors='ignore')
print(posthoc_df)

print("\n==============================")
print("APA STYLE REPORT")
print("==============================")

if p_pair < 0.001:
    p_text = "p < .001"
else:
    p_text = f"p = {p_pair:.6e}"

print(
    f"A post-hoc Pearson chi-square test comparing "
    f"Coded_Teacher and Affective_Teacher "
    f"showed that subskill distributions differed "
    f"significantly, χ²({dof_pair}) = "
    f"{chi2_pair:.2f}, {p }{p_text}."
)


FULL CONTINGENCY TABLE
     Affective_Coded  Affective_Predict  Affective_Teacher
AS1                0                  0                 58
AS2               18                 19                 42
AS3                1                  0                 10

OVERALL CHI-SQUARE TEST
Chi-square = 40.5603
Degrees of freedom = 4
P-value = 3.314418e-08

POST-HOC: Coded vs Teacher
     Affective_Coded  Affective_Teacher
AS1                0                 58
AS2               18                 42
AS3                1                 10

Chi-square = 21.4379
Degrees of freedom = 2
P-value = 2.212170e-05
[[ 8.54263566 49.45736434]
 [ 8.8372093  51.1627907 ]
 [ 1.62015504  9.37984496]]

STANDARDIZED RESIDUALS
     Affective_Coded  Affective_Teacher
AS1            -2.92               1.21
AS2             3.08              -1.28
AS3            -0.49               0.20

CELLS WITH LARGE RESIDUALS
AS1 in Affective_Coded: residual = -2.92 (lower than expected)
AS2 in Affective_Coded: residual = 